# 03. Guardrails and Human in the Loop

This notebook shows how to validate outputs and route uncertain cases to a person. That is a core pattern for research workflows where ambiguity matters.

## Concepts
- Guardrails
- Human review
- Uncertainty thresholds
- Safer automation

## Dataset
The notebook uses a small set of ambiguous archival-style records.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from common import LETTER_TEXTS, ReviewDecision
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput

records = LETTER_TEXTS
records

## Step 1: A tiny review heuristic

We treat low confidence or uncertain OCR as a signal for human review.

In [ ]:
def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

sample_decision = ReviewDecision(record_id=3, uncertain=True, confidence=0.52, notes='OCR ambiguity around place name')
should_review(sample_decision)

## Step 2: Build a guardrail

A guardrail can reject inputs that clearly need human review before the agent continues.

In [ ]:
def uncertainty_guardrail(ctx, agent, input_text):
    if isinstance(input_text, str) and ('may be' in input_text.lower() or 'unclear' in input_text.lower()):
        return GuardrailFunctionOutput(output_info='uncertain text', tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
guardrail

## Step 3: Agent with guardrail

If the text looks uncertain, the agent should stop and let the person review it.

In [ ]:
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
)

review_agent

## Step 4: Run on a clean and an uncertain record

The clean record should pass. The uncertain one should trigger the guardrail.

In [ ]:
clean_result = Runner.run_sync(review_agent, records[0]['text'])
print(clean_result.final_output)

uncertain_text = 'The scan is unclear; the place may be Seville or Sevilla.'
try:
    uncertain_result = Runner.run_sync(review_agent, uncertain_text)
    print(uncertain_result.final_output)
except Exception as exc:
    print(type(exc).__name__, exc)

## Human-in-the-loop checkpoint

If the guardrail triggers, the next step is not to force a guess. It is to ask a person to resolve the ambiguity and then resume.

## Exercise

Change the threshold so only records with both OCR ambiguity and low confidence are sent to review.

## Solution

Modify `uncertainty_guardrail` to check for two conditions at once, for example `('unclear' in text and confidence < 0.8)`.